In [ ]:
CREATE OR REPLACE TABLE IOWA_HISTORICAL_MARKET_RISK AS
WITH yearly AS (
    SELECT
        'Statewide Iowa' AS REGION,
        YEAR,
        AVG(PRICE_USD_PER_BUSHEL) AS AVG_PRICE_USD_PER_BUSHEL,
        AVG(YIELD_BU_PER_ACRE) AS AVG_YIELD_BU_PER_ACRE
    FROM IOWA_MARKET_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY YEAR
),
norm AS (
    SELECT *,
        (AVG_PRICE_USD_PER_BUSHEL - MIN(AVG_PRICE_USD_PER_BUSHEL) OVER ()) /
        NULLIF(MAX(AVG_PRICE_USD_PER_BUSHEL) OVER () - MIN(AVG_PRICE_USD_PER_BUSHEL) OVER (),0) AS PRICE_NORM,

        (AVG_YIELD_BU_PER_ACRE - MIN(AVG_YIELD_BU_PER_ACRE) OVER ()) /
        NULLIF(MAX(AVG_YIELD_BU_PER_ACRE) OVER () - MIN(AVG_YIELD_BU_PER_ACRE) OVER (),0) AS YIELD_NORM
    FROM yearly
),
risked AS (
    SELECT *,
        ROUND((1 - ((0.70 * PRICE_NORM) + (0.30 * YIELD_NORM))) * 99, 2) AS MARKET_RISK_SCORE
    FROM norm
)
SELECT *,
    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN 'Very Low'
        WHEN MARKET_RISK_SCORE < 40 THEN 'Low'
        WHEN MARKET_RISK_SCORE < 60 THEN 'Moderate'
        WHEN MARKET_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS MARKET_RISK_LEVEL,

    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' shows exceptionally strong market conditions. Average crop price was $' || ROUND(AVG_PRICE_USD_PER_BUSHEL,2) || ' per bushel and average yield was ' || ROUND(AVG_YIELD_BU_PER_ACRE,1) || ' bushels per acre, suggesting strong revenue potential and very limited market pressure.'
        WHEN MARKET_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' shows favorable market conditions. Average crop price was $' || ROUND(AVG_PRICE_USD_PER_BUSHEL,2) || ' per bushel and yield was ' || ROUND(AVG_YIELD_BU_PER_ACRE,1) || ' bushels per acre, supporting stable returns.'
        WHEN MARKET_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' reflects moderate market risk. Price and yield conditions suggest a balanced but uncertain market environment.'
        WHEN MARKET_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' shows high market risk. Weaker price or yield conditions may tighten margins and increase exposure to market stress.'
        ELSE REGION || ' in ' || YEAR || ' shows very high market risk. Weak price and yield conditions suggest elevated financial pressure and a less favorable market outlook.'
    END AS MARKET_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_HISTORICAL_MARKET_RISK
ORDER BY YEAR;

In [ ]:
CREATE OR REPLACE TABLE IOWA_HISTORICAL_WEATHER_RISK AS
WITH yearly AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Iowa'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Iowa'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Iowa'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Iowa'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Iowa'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,
        YEAR,
        AVG(PRECIPITATION_IN_INCHES) AS AVG_PRECIP,
        AVG(AVERAGE_TEMPERATURE) AS AVG_TEMP,
        AVG(PALMER_DROUGHT_SEVERITY_INDEX) AS AVG_DROUGHT,
        AVG(HEATING_DEGREE_DAYS) AS AVG_HEAT,
        AVG(COOLING_DEGREE_DAYS) AS AVG_COOL
    FROM IOWA_WEATHER_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY 1, YEAR
),
norm AS (
    SELECT *,
        1 - ((AVG_PRECIP - MIN(AVG_PRECIP) OVER ()) /
        NULLIF(MAX(AVG_PRECIP) OVER () - MIN(AVG_PRECIP) OVER (), 0)) AS PRECIP_RISK,

        ((AVG_TEMP - MIN(AVG_TEMP) OVER ()) /
        NULLIF(MAX(AVG_TEMP) OVER () - MIN(AVG_TEMP) OVER (), 0)) AS TEMP_RISK,

        ((AVG_DROUGHT - MIN(AVG_DROUGHT) OVER ()) /
        NULLIF(MAX(AVG_DROUGHT) OVER () - MIN(AVG_DROUGHT) OVER (), 0)) AS DROUGHT_RISK,

        ((AVG_HEAT - MIN(AVG_HEAT) OVER ()) /
        NULLIF(MAX(AVG_HEAT) OVER () - MIN(AVG_HEAT) OVER (), 0)) AS HEAT_RISK,

        ((AVG_COOL - MIN(AVG_COOL) OVER ()) /
        NULLIF(MAX(AVG_COOL) OVER () - MIN(AVG_COOL) OVER (), 0)) AS COOL_RISK
    FROM yearly
),
risked AS (
    SELECT *,
        ROUND(
            (
                0.30 * COALESCE(PRECIP_RISK, 0) +
                0.30 * COALESCE(DROUGHT_RISK, 0) +
                0.15 * COALESCE(TEMP_RISK, 0) +
                0.125 * COALESCE(HEAT_RISK, 0) +
                0.125 * COALESCE(COOL_RISK, 0)
            ) * 99,
            2
        ) AS WEATHER_RISK_SCORE
    FROM norm
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY WEATHER_RISK_SCORE DESC, REGION) AS WEATHER_RISK_RANK,

    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN 'Very Low'
        WHEN WEATHER_RISK_SCORE < 40 THEN 'Low'
        WHEN WEATHER_RISK_SCORE < 60 THEN 'Moderate'
        WHEN WEATHER_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS WEATHER_RISK_LEVEL,

    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' shows very low weather risk. Precipitation averaged ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature averaged ' || ROUND(AVG_TEMP, 2) || ', and drought index averaged ' || ROUND(AVG_DROUGHT, 2) || ', suggesting limited environmental stress.'
        WHEN WEATHER_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' shows low weather risk. Precipitation averaged ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature averaged ' || ROUND(AVG_TEMP, 2) || ', and drought index averaged ' || ROUND(AVG_DROUGHT, 2) || ', suggesting manageable growing conditions.'
        WHEN WEATHER_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' shows moderate weather risk. Precipitation averaged ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature averaged ' || ROUND(AVG_TEMP, 2) || ', and drought index averaged ' || ROUND(AVG_DROUGHT, 2) || ', suggesting some environmental uncertainty.'
        WHEN WEATHER_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' shows high weather risk. Precipitation averaged ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature averaged ' || ROUND(AVG_TEMP, 2) || ', and drought index averaged ' || ROUND(AVG_DROUGHT, 2) || ', indicating elevated climate stress.'
        ELSE REGION || ' in ' || YEAR || ' shows very high weather risk. Precipitation averaged ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature averaged ' || ROUND(AVG_TEMP, 2) || ', and drought index averaged ' || ROUND(AVG_DROUGHT, 2) || ', suggesting difficult environmental conditions for agriculture.'
    END AS WEATHER_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_HISTORICAL_WEATHER_RISK
ORDER BY YEAR, WEATHER_RISK_RANK;

In [ ]:
CREATE OR REPLACE TABLE IOWA_HISTORICAL_LAND_RISK AS
WITH yearly AS (
    SELECT
        CASE
            WHEN UPPER(REGION) LIKE '%NORTHWEST%' THEN 'Northwest Iowa'
            WHEN UPPER(REGION) LIKE '%NORTHEAST%' THEN 'Northeast Iowa'
            WHEN UPPER(REGION) LIKE '%SOUTHWEST%' THEN 'Southwest Iowa'
            WHEN UPPER(REGION) LIKE '%SOUTHEAST%' THEN 'Southeast Iowa'
            WHEN UPPER(REGION) LIKE '%CENTRAL%' THEN 'Central Iowa'
            ELSE INITCAP(TRIM(REGION))
        END AS REGION,
        YEAR,
        AVG(LAND_VALUE_AVG_PER_ACRE) AS AVG_LAND_VALUE,
        AVG(ALL_CROPLAND_PER_ACRE) AS AVG_CROPLAND_VALUE,
        AVG(FARM_REAL_ESTATE_PER_ACRE) AS AVG_FARM_RE_VALUE,
        AVG(CASH_RENT_DRYLAND_PER_ACRE) AS AVG_RENT_DRYLAND,
        AVG(CASH_RENT_IRRIGATED_PER_ACRE) AS AVG_RENT_IRRIGATED,
        AVG(CASH_RENT_PASTURE_PER_ACRE) AS AVG_RENT_PASTURE
    FROM IOWA_LAND_VALUE_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY 1, YEAR
),
factor AS (
    SELECT *,
        ROUND(
            COALESCE(AVG_LAND_VALUE, 0) * 0.40 +
            COALESCE(AVG_CROPLAND_VALUE, 0) * 0.20 +
            COALESCE(AVG_FARM_RE_VALUE, 0) * 0.15 +
            ((COALESCE(AVG_RENT_DRYLAND, 0) + COALESCE(AVG_RENT_IRRIGATED, 0) + COALESCE(AVG_RENT_PASTURE, 0)) / 3) * 0.25,
            2
        ) AS LAND_FACTOR_VALUE
    FROM yearly
),
risked AS (
    SELECT *,
        ROUND(
            99 * ((LAND_FACTOR_VALUE - MIN(LAND_FACTOR_VALUE) OVER ()) /
            NULLIF(MAX(LAND_FACTOR_VALUE) OVER () - MIN(LAND_FACTOR_VALUE) OVER (), 0)),
            2
        ) AS LAND_RISK_SCORE
    FROM factor
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY LAND_RISK_SCORE DESC, REGION) AS LAND_RISK_RANK,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN 'Very Low'
        WHEN LAND_RISK_SCORE < 40 THEN 'Low'
        WHEN LAND_RISK_SCORE < 60 THEN 'Moderate'
        WHEN LAND_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS LAND_RISK_LEVEL,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' shows very low land risk. Average land value was $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, cropland value was $' || ROUND(AVG_CROPLAND_VALUE, 2) || ', and farm real estate value was $' || ROUND(AVG_FARM_RE_VALUE, 2) || ', suggesting manageable ownership and operating cost pressure.'
        WHEN LAND_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' shows low land risk. Average land value was $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre and rent levels remained relatively stable, supporting manageable production costs.'
        WHEN LAND_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' shows moderate land risk. Average land value was $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, cropland value was $' || ROUND(AVG_CROPLAND_VALUE, 2) || ', and rental pressure became more noticeable.'
        WHEN LAND_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' shows high land risk. Elevated land values around $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre and rental costs suggest tighter margins and greater financial pressure.'
        ELSE REGION || ' in ' || YEAR || ' shows very high land risk. High land values around $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, real estate costs, and rental pressure suggest a costly operating environment with significant financial exposure.'
    END AS LAND_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_HISTORICAL_LAND_RISK
ORDER BY YEAR, LAND_RISK_RANK;

In [ ]:
CREATE OR REPLACE TABLE IOWA_HISTORICAL_TOTAL_RISK AS
WITH combined AS (
    SELECT
        w.REGION,
        w.YEAR,
        m.MARKET_RISK_SCORE,
        w.WEATHER_RISK_SCORE,
        l.LAND_RISK_SCORE,
        ROUND(
            0.40 * m.MARKET_RISK_SCORE +
            0.35 * w.WEATHER_RISK_SCORE +
            0.25 * l.LAND_RISK_SCORE,
            2
        ) AS TOTAL_RISK_SCORE
    FROM IOWA_HISTORICAL_WEATHER_RISK w
    JOIN IOWA_HISTORICAL_LAND_RISK l
        ON w.REGION = l.REGION
       AND w.YEAR = l.YEAR
    JOIN IOWA_HISTORICAL_MARKET_RISK m
        ON w.YEAR = m.YEAR
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY TOTAL_RISK_SCORE DESC, REGION) AS TOTAL_RISK_RANK
    FROM combined
)
SELECT
    *,
    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN 'Very Low'
        WHEN TOTAL_RISK_SCORE < 40 THEN 'Low'
        WHEN TOTAL_RISK_SCORE < 60 THEN 'Moderate'
        WHEN TOTAL_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS TOTAL_RISK_LEVEL,

    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' shows very low total agricultural risk. Market, weather, and land conditions collectively suggest a favorable environment with limited financial and environmental pressure.'
        WHEN TOTAL_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' shows low total agricultural risk. The combined score reflects manageable market, weather, and land conditions.'
        WHEN TOTAL_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' shows moderate total agricultural risk. Market risk was ' || MARKET_RISK_SCORE || ', weather risk was ' || WEATHER_RISK_SCORE || ', and land risk was ' || LAND_RISK_SCORE || ', suggesting a balanced but uncertain risk environment.'
        WHEN TOTAL_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' shows high total agricultural risk. Market risk was ' || MARKET_RISK_SCORE || ', weather risk was ' || WEATHER_RISK_SCORE || ', and land risk was ' || LAND_RISK_SCORE || ', showing stronger pressure from one or more risk categories.'
        ELSE REGION || ' in ' || YEAR || ' shows very high total agricultural risk. Market risk (' || MARKET_RISK_SCORE || '), weather risk (' || WEATHER_RISK_SCORE || '), and land risk (' || LAND_RISK_SCORE || ') combine to suggest elevated financial and production pressure.'
    END AS TOTAL_ADVANCED_DESCRIPTION
FROM ranked;

In [ ]:
SELECT *
FROM IOWA_HISTORICAL_TOTAL_RISK
ORDER BY YEAR, TOTAL_RISK_RANK;

In [ ]:
CREATE OR REPLACE TABLE IOWA_FUTURE_MARKET_RISK_2025_2030 AS
WITH forecast_years AS (
    SELECT 2025 AS YEAR UNION ALL
    SELECT 2026 UNION ALL
    SELECT 2027 UNION ALL
    SELECT 2028 UNION ALL
    SELECT 2029 UNION ALL
    SELECT 2030
),
yearly AS (
    SELECT
        YEAR,
        AVG(PRICE_USD_PER_BUSHEL) AS AVG_PRICE_USD_PER_BUSHEL,
        AVG(YIELD_BU_PER_ACRE) AS AVG_YIELD_BU_PER_ACRE
    FROM IOWA_MARKET_DATA
    WHERE YEAR BETWEEN 2014 AND 2024
    GROUP BY YEAR
),
model AS (
    SELECT
        REGR_SLOPE(AVG_PRICE_USD_PER_BUSHEL, YEAR) AS PRICE_SLOPE,
        REGR_INTERCEPT(AVG_PRICE_USD_PER_BUSHEL, YEAR) AS PRICE_INTERCEPT,
        REGR_SLOPE(AVG_YIELD_BU_PER_ACRE, YEAR) AS YIELD_SLOPE,
        REGR_INTERCEPT(AVG_YIELD_BU_PER_ACRE, YEAR) AS YIELD_INTERCEPT
    FROM yearly
),
predicted AS (
    SELECT
        'Statewide Iowa' AS REGION,
        f.YEAR,
        m.PRICE_INTERCEPT + m.PRICE_SLOPE * f.YEAR AS PREDICTED_PRICE_USD_PER_BUSHEL,
        m.YIELD_INTERCEPT + m.YIELD_SLOPE * f.YEAR AS PREDICTED_YIELD_BU_PER_ACRE
    FROM forecast_years f
    CROSS JOIN model m
),
norm AS (
    SELECT *,
        (PREDICTED_PRICE_USD_PER_BUSHEL - MIN(PREDICTED_PRICE_USD_PER_BUSHEL) OVER ()) /
        NULLIF(MAX(PREDICTED_PRICE_USD_PER_BUSHEL) OVER () - MIN(PREDICTED_PRICE_USD_PER_BUSHEL) OVER (), 0) AS PRICE_NORM,

        (PREDICTED_YIELD_BU_PER_ACRE - MIN(PREDICTED_YIELD_BU_PER_ACRE) OVER ()) /
        NULLIF(MAX(PREDICTED_YIELD_BU_PER_ACRE) OVER () - MIN(PREDICTED_YIELD_BU_PER_ACRE) OVER (), 0) AS YIELD_NORM
    FROM predicted
),
risked AS (
    SELECT *,
        ROUND((1 - ((0.70 * PRICE_NORM) + (0.30 * YIELD_NORM))) * 99, 2) AS MARKET_RISK_SCORE
    FROM norm
)
SELECT
    *,
    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN 'Very Low'
        WHEN MARKET_RISK_SCORE < 40 THEN 'Low'
        WHEN MARKET_RISK_SCORE < 60 THEN 'Moderate'
        WHEN MARKET_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS MARKET_RISK_LEVEL,

    CASE
        WHEN MARKET_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' is projected to show exceptionally strong market conditions. Forecasted price is $' || ROUND(PREDICTED_PRICE_USD_PER_BUSHEL, 2) || ' per bushel and projected yield is ' || ROUND(PREDICTED_YIELD_BU_PER_ACRE, 1) || ' bushels per acre, suggesting strong revenue potential.'
        WHEN MARKET_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' is projected to show favorable market conditions. Forecasted price is $' || ROUND(PREDICTED_PRICE_USD_PER_BUSHEL, 2) || ' per bushel and projected yield is ' || ROUND(PREDICTED_YIELD_BU_PER_ACRE, 1) || ' bushels per acre.'
        WHEN MARKET_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' is projected to show moderate market risk. Forecasted price is $' || ROUND(PREDICTED_PRICE_USD_PER_BUSHEL, 2) || ' per bushel and projected yield is ' || ROUND(PREDICTED_YIELD_BU_PER_ACRE, 1) || ' bushels per acre.'
        WHEN MARKET_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' is projected to show high market risk. Forecasted price is $' || ROUND(PREDICTED_PRICE_USD_PER_BUSHEL, 2) || ' per bushel and projected yield is ' || ROUND(PREDICTED_YIELD_BU_PER_ACRE, 1) || ' bushels per acre.'
        ELSE REGION || ' in ' || YEAR || ' is projected to show very high market risk. Forecasted price is $' || ROUND(PREDICTED_PRICE_USD_PER_BUSHEL, 2) || ' per bushel and projected yield is ' || ROUND(PREDICTED_YIELD_BU_PER_ACRE, 1) || ' bushels per acre.'
    END AS MARKET_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_FUTURE_MARKET_RISK_2025_2030
ORDER BY YEAR;

In [ ]:
CREATE OR REPLACE TABLE IOWA_FUTURE_WEATHER_RISK_2025_2030 AS
WITH forecast_years AS (
    SELECT 2025 AS YEAR UNION ALL
    SELECT 2026 UNION ALL
    SELECT 2027 UNION ALL
    SELECT 2028 UNION ALL
    SELECT 2029 UNION ALL
    SELECT 2030
),
historical AS (
    SELECT REGION, YEAR, AVG_PRECIP, AVG_TEMP, AVG_DROUGHT, AVG_HEAT, AVG_COOL
    FROM IOWA_HISTORICAL_WEATHER_RISK
),
long_data AS (
    SELECT REGION, YEAR, 'PRECIP' AS METRIC, AVG_PRECIP AS VALUE FROM historical
    UNION ALL SELECT REGION, YEAR, 'TEMP', AVG_TEMP FROM historical
    UNION ALL SELECT REGION, YEAR, 'DROUGHT', AVG_DROUGHT FROM historical
    UNION ALL SELECT REGION, YEAR, 'HEAT', AVG_HEAT FROM historical
    UNION ALL SELECT REGION, YEAR, 'COOL', AVG_COOL FROM historical
),
model AS (
    SELECT
        REGION,
        METRIC,
        REGR_SLOPE(VALUE, YEAR) AS SLOPE,
        REGR_INTERCEPT(VALUE, YEAR) AS INTERCEPT
    FROM long_data
    GROUP BY REGION, METRIC
),
predicted AS (
    SELECT
        m.REGION,
        f.YEAR,
        m.METRIC,
        m.INTERCEPT + m.SLOPE * f.YEAR AS PREDICTED_VALUE
    FROM model m
    CROSS JOIN forecast_years f
),
pivoted AS (
    SELECT
        REGION,
        YEAR,
        AVG(CASE WHEN METRIC = 'PRECIP' THEN PREDICTED_VALUE END) AS AVG_PRECIP,
        AVG(CASE WHEN METRIC = 'TEMP' THEN PREDICTED_VALUE END) AS AVG_TEMP,
        AVG(CASE WHEN METRIC = 'DROUGHT' THEN PREDICTED_VALUE END) AS AVG_DROUGHT,
        AVG(CASE WHEN METRIC = 'HEAT' THEN PREDICTED_VALUE END) AS AVG_HEAT,
        AVG(CASE WHEN METRIC = 'COOL' THEN PREDICTED_VALUE END) AS AVG_COOL
    FROM predicted
    GROUP BY REGION, YEAR
),
norm AS (
    SELECT *,
        1 - ((AVG_PRECIP - MIN(AVG_PRECIP) OVER ()) /
        NULLIF(MAX(AVG_PRECIP) OVER () - MIN(AVG_PRECIP) OVER (), 0)) AS PRECIP_RISK,

        ((AVG_TEMP - MIN(AVG_TEMP) OVER ()) /
        NULLIF(MAX(AVG_TEMP) OVER () - MIN(AVG_TEMP) OVER (), 0)) AS TEMP_RISK,

        ((AVG_DROUGHT - MIN(AVG_DROUGHT) OVER ()) /
        NULLIF(MAX(AVG_DROUGHT) OVER () - MIN(AVG_DROUGHT) OVER (), 0)) AS DROUGHT_RISK,

        ((AVG_HEAT - MIN(AVG_HEAT) OVER ()) /
        NULLIF(MAX(AVG_HEAT) OVER () - MIN(AVG_HEAT) OVER (), 0)) AS HEAT_RISK,

        ((AVG_COOL - MIN(AVG_COOL) OVER ()) /
        NULLIF(MAX(AVG_COOL) OVER () - MIN(AVG_COOL) OVER (), 0)) AS COOL_RISK
    FROM pivoted
),
risked AS (
    SELECT *,
        ROUND(
            (
                0.30 * COALESCE(PRECIP_RISK, 0) +
                0.30 * COALESCE(DROUGHT_RISK, 0) +
                0.15 * COALESCE(TEMP_RISK, 0) +
                0.125 * COALESCE(HEAT_RISK, 0) +
                0.125 * COALESCE(COOL_RISK, 0)
            ) * 99,
            2
        ) AS WEATHER_RISK_SCORE
    FROM norm
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY WEATHER_RISK_SCORE DESC, REGION) AS WEATHER_RISK_RANK,

    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN 'Very Low'
        WHEN WEATHER_RISK_SCORE < 40 THEN 'Low'
        WHEN WEATHER_RISK_SCORE < 60 THEN 'Moderate'
        WHEN WEATHER_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS WEATHER_RISK_LEVEL,

    CASE
        WHEN WEATHER_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' is projected to show very low weather risk. Forecasted precipitation is ' || ROUND(AVG_PRECIP, 2) || ' inches, temperature is ' || ROUND(AVG_TEMP, 2) || ', and drought index is ' || ROUND(AVG_DROUGHT, 2) || '.'
        WHEN WEATHER_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' is projected to show low weather risk. Forecasted precipitation, temperature, and drought conditions remain manageable.'
        WHEN WEATHER_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' is projected to show moderate weather risk. Forecasted precipitation is ' || ROUND(AVG_PRECIP, 2) || ' inches and drought index is ' || ROUND(AVG_DROUGHT, 2) || '.'
        WHEN WEATHER_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' is projected to show high weather risk. Forecasted weather conditions suggest elevated crop stress.'
        ELSE REGION || ' in ' || YEAR || ' is projected to show very high weather risk. Forecasted drought, precipitation, and temperature conditions suggest difficult environmental pressure.'
    END AS WEATHER_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_FUTURE_WEATHER_RISK_2025_2030
ORDER BY YEAR, WEATHER_RISK_RANK;

In [ ]:
CREATE OR REPLACE TABLE IOWA_FUTURE_LAND_RISK_2025_2030 AS
WITH forecast_years AS (
    SELECT 2025 AS YEAR UNION ALL
    SELECT 2026 UNION ALL
    SELECT 2027 UNION ALL
    SELECT 2028 UNION ALL
    SELECT 2029 UNION ALL
    SELECT 2030
),
historical AS (
    SELECT
        REGION,
        YEAR,
        AVG_LAND_VALUE,
        AVG_CROPLAND_VALUE,
        AVG_FARM_RE_VALUE,
        AVG_RENT_DRYLAND,
        AVG_RENT_IRRIGATED,
        AVG_RENT_PASTURE
    FROM IOWA_HISTORICAL_LAND_RISK
),
long_data AS (
    SELECT REGION, YEAR, 'LAND' AS METRIC, AVG_LAND_VALUE AS VALUE FROM historical
    UNION ALL SELECT REGION, YEAR, 'CROPLAND', AVG_CROPLAND_VALUE FROM historical
    UNION ALL SELECT REGION, YEAR, 'FARM_RE', AVG_FARM_RE_VALUE FROM historical
    UNION ALL SELECT REGION, YEAR, 'RENT_DRYLAND', AVG_RENT_DRYLAND FROM historical
    UNION ALL SELECT REGION, YEAR, 'RENT_IRRIGATED', AVG_RENT_IRRIGATED FROM historical
    UNION ALL SELECT REGION, YEAR, 'RENT_PASTURE', AVG_RENT_PASTURE FROM historical
),
model AS (
    SELECT
        REGION,
        METRIC,
        REGR_SLOPE(VALUE, YEAR) AS SLOPE,
        REGR_INTERCEPT(VALUE, YEAR) AS INTERCEPT
    FROM long_data
    GROUP BY REGION, METRIC
),
predicted AS (
    SELECT
        m.REGION,
        f.YEAR,
        m.METRIC,
        m.INTERCEPT + m.SLOPE * f.YEAR AS PREDICTED_VALUE
    FROM model m
    CROSS JOIN forecast_years f
),
pivoted AS (
    SELECT
        REGION,
        YEAR,
        AVG(CASE WHEN METRIC = 'LAND' THEN PREDICTED_VALUE END) AS AVG_LAND_VALUE,
        AVG(CASE WHEN METRIC = 'CROPLAND' THEN PREDICTED_VALUE END) AS AVG_CROPLAND_VALUE,
        AVG(CASE WHEN METRIC = 'FARM_RE' THEN PREDICTED_VALUE END) AS AVG_FARM_RE_VALUE,
        AVG(CASE WHEN METRIC = 'RENT_DRYLAND' THEN PREDICTED_VALUE END) AS AVG_RENT_DRYLAND,
        AVG(CASE WHEN METRIC = 'RENT_IRRIGATED' THEN PREDICTED_VALUE END) AS AVG_RENT_IRRIGATED,
        AVG(CASE WHEN METRIC = 'RENT_PASTURE' THEN PREDICTED_VALUE END) AS AVG_RENT_PASTURE
    FROM predicted
    GROUP BY REGION, YEAR
),
factor AS (
    SELECT *,
        ROUND(
            COALESCE(AVG_LAND_VALUE, 0) * 0.40 +
            COALESCE(AVG_CROPLAND_VALUE, 0) * 0.20 +
            COALESCE(AVG_FARM_RE_VALUE, 0) * 0.15 +
            ((COALESCE(AVG_RENT_DRYLAND, 0) + COALESCE(AVG_RENT_IRRIGATED, 0) + COALESCE(AVG_RENT_PASTURE, 0)) / 3) * 0.25,
            2
        ) AS LAND_FACTOR_VALUE
    FROM pivoted
),
risked AS (
    SELECT *,
        ROUND(
            99 * ((LAND_FACTOR_VALUE - MIN(LAND_FACTOR_VALUE) OVER ()) /
            NULLIF(MAX(LAND_FACTOR_VALUE) OVER () - MIN(LAND_FACTOR_VALUE) OVER (), 0)),
            2
        ) AS LAND_RISK_SCORE
    FROM factor
)
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY LAND_RISK_SCORE DESC, REGION) AS LAND_RISK_RANK,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN 'Very Low'
        WHEN LAND_RISK_SCORE < 40 THEN 'Low'
        WHEN LAND_RISK_SCORE < 60 THEN 'Moderate'
        WHEN LAND_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS LAND_RISK_LEVEL,

    CASE
        WHEN LAND_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' is projected to show very low land risk. Forecasted land value is $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, suggesting manageable future ownership and operating costs.'
        WHEN LAND_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' is projected to show low land risk. Forecasted land and rent values remain relatively stable.'
        WHEN LAND_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' is projected to show moderate land risk. Forecasted land value is $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, suggesting land costs may become more noticeable.'
        WHEN LAND_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' is projected to show high land risk. Forecasted land value is $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre with stronger cost pressure.'
        ELSE REGION || ' in ' || YEAR || ' is projected to show very high land risk. Forecasted land value is $' || ROUND(AVG_LAND_VALUE, 2) || ' per acre, indicating significant ownership and operating cost exposure.'
    END AS LAND_ADVANCED_DESCRIPTION
FROM risked;

In [ ]:
SELECT *
FROM IOWA_FUTURE_LAND_RISK_2025_2030
ORDER BY YEAR, LAND_RISK_RANK;

In [ ]:
CREATE OR REPLACE TABLE IOWA_FUTURE_TOTAL_RISK_2025_2030 AS
WITH combined AS (
    SELECT
        w.REGION,
        w.YEAR,
        m.MARKET_RISK_SCORE,
        w.WEATHER_RISK_SCORE,
        l.LAND_RISK_SCORE,
        ROUND(
            0.40 * m.MARKET_RISK_SCORE +
            0.35 * w.WEATHER_RISK_SCORE +
            0.25 * l.LAND_RISK_SCORE,
            2
        ) AS TOTAL_RISK_SCORE
    FROM IOWA_FUTURE_WEATHER_RISK_2025_2030 w
    JOIN IOWA_FUTURE_LAND_RISK_2025_2030 l
        ON w.REGION = l.REGION
       AND w.YEAR = l.YEAR
    JOIN IOWA_FUTURE_MARKET_RISK_2025_2030 m
        ON w.YEAR = m.YEAR
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY YEAR ORDER BY TOTAL_RISK_SCORE DESC, REGION) AS TOTAL_RISK_RANK
    FROM combined
)
SELECT
    *,
    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN 'Very Low'
        WHEN TOTAL_RISK_SCORE < 40 THEN 'Low'
        WHEN TOTAL_RISK_SCORE < 60 THEN 'Moderate'
        WHEN TOTAL_RISK_SCORE < 80 THEN 'High'
        ELSE 'Very High'
    END AS TOTAL_RISK_LEVEL,

    CASE
        WHEN TOTAL_RISK_SCORE < 20 THEN REGION || ' in ' || YEAR || ' is projected to show very low total agricultural risk. Market, weather, and land conditions collectively suggest a favorable outlook.'
        WHEN TOTAL_RISK_SCORE < 40 THEN REGION || ' in ' || YEAR || ' is projected to show low total agricultural risk. The combined model suggests manageable market, weather, and land pressure.'
        WHEN TOTAL_RISK_SCORE < 60 THEN REGION || ' in ' || YEAR || ' is projected to show moderate total agricultural risk. Market risk is ' || MARKET_RISK_SCORE || ', weather risk is ' || WEATHER_RISK_SCORE || ', and land risk is ' || LAND_RISK_SCORE || '.'
        WHEN TOTAL_RISK_SCORE < 80 THEN REGION || ' in ' || YEAR || ' is projected to show high total agricultural risk. The weighted model indicates stronger concern from market, weather, or land pressure.'
        ELSE REGION || ' in ' || YEAR || ' is projected to show very high total agricultural risk. Market risk (' || MARKET_RISK_SCORE || '), weather risk (' || WEATHER_RISK_SCORE || '), and land risk (' || LAND_RISK_SCORE || ') combine to suggest elevated financial and production pressure.'
    END AS TOTAL_ADVANCED_DESCRIPTION
FROM ranked;

In [ ]:
SELECT *
FROM IOWA_FUTURE_TOTAL_RISK_2025_2030
ORDER BY YEAR, TOTAL_RISK_RANK;